# Preparing supply-use data for CVXLab

Prepare Italy's 2022 supply-use data for use in a CVXLab model.
Download the official ISTAT workbooks, attach compatible satellite accounts,
translate the labels into English, aggregate and inspect the database, then
export the matrices and sets to CVXLab.

This example uses the published national classifications. Electricity, gas,
steam and air conditioning remain one activity; no electricity-technology
disaggregation or power-sector scenarios are introduced.

## Problem statement

Prepare a national supply-use database with domestic uses, separate product
imports and compatible emissions, employment and energy accounts for a
CVXLab model.

Start from 63 ISTAT activities and products and obtain 15 of each, retaining
7 final-demand categories, 8 income/accounting rows and 27 satellite accounts.
Inspect the database and export its baseline matrices using the CVXLab model's
table names, coordinates and filters.

## Tutorial materials

Download this notebook, `aggregation.xlsx` and
`cvxlab_model.zip` from the [tutorial page](https://tutorials.marioproject.com/preparing-supply-use-data-for-cvxlab/index.html#tutorial-materials).
Extract the archive beside the notebook to obtain `cvxlab_model/`.
It contains the CVXLab `model_settings.xlsx`, classified sets
and input tables. The notebook regenerates
those inputs from the official source data.

The ISTAT parser additions are intended for MARIO 1.3.6 and are not in 1.3.5.
Install the development snapshot below in a dedicated Python 3.12
environment, then restart the notebook kernel after installation.

```sh
conda create -n mario_cvxlab_tutorial python=3.12 pip
conda activate mario_cvxlab_tutorial
python -m pip install "mariopy @ git+https://github.com/it-is-me-mario/MARIO.git@e786e43fd01186bf1707cf098e2ceea1afb29e49" "cvxlab==1.0.1b1" jupyterlab
python -m jupyter lab
```

The code uses relative paths from the working folder. The official monetary
release and the selected Eurostat snapshots are cached under `source_data/`.
The archive's `README.rst` describes the data sources and model input tables. Source data is downloaded separately rather than bundled here.

## Database download

Download the official [ISTAT 2020–2022 edition](https://www.istat.it/tavole-di-dati/il-sistema-di-tavole-input-output-anni-2020-2022/).
The parser selects 2022 at current basic prices. The native classification
contains 63 NACE Rev. 2 branches and 63 CPA 2.1 products, excluding
extraterritorial organisations. Existing cached workbooks are reused.

In [ ]:
import mario

mario.download_istat_io(path="source_data", edition="2020-2022")

## Database parsing

Read SUPPLY, USE and IMPORT and select the three optional satellite
groups. `download=True` reuses the monetary release and fetches missing
Eurostat snapshots for Italy in the same reference year.

ISTAT is already single-region: do not call `to_single_region()`.
`IMP_a` and `IMP_Y` retain imports by product and purchasing activity or final
demand. `U` and `Yc` contain domestic uses; no origin-country detail is available.

The selected accounts are [AEA emissions](https://ec.europa.eu/eurostat/databrowser/view/env_ac_ainah_r2/default/table?lang=en),
[employment](https://ec.europa.eu/eurostat/databrowser/view/nama_10_a64_e/default/table?lang=en)
and [PEFA net domestic energy use](https://ec.europa.eu/eurostat/databrowser/view/env_ac_pefa04/default/table?lang=en).
Hours worked come from the ISTAT USE workbook. Production accounts populate
`Ea`; direct household emissions and energy populate `EY`. `Ec` is zero.
Native NACE codes align the satellite observations before labels are translated.
Missing required observations cause an error rather than being filled with zero.

The variable `db` is a `mario.Database` object: it contains the SUT,
classifications, units, metadata and scenarios. The parsed matrix values
belong to its `baseline` scenario, which is used throughout this example.
See [Objects in MARIO](https://www.marioproject.com/en/latest/get_started/table_formats.html#objects-in-mario)
for the distinction between a database, a table, a scenario and a matrix.



In [ ]:
db = mario.parse_istat(
    path="source_data",
    year=2022,
    table="SUT",
    level="63",
    price="current",
    valuation="basic",
    import_representation="separate",
    extensions=["emissions", "employment", "energy"],
    download=True,
    edition="2020-2022",
)

## Aggregation

`aggregation.xlsx` maps the original ISTAT labels directly to the English
classifications used by the model. Its Activity and Commodity sheets aggregate
63 branches and products into 15 groups. Consumption category, Factor of
production and Satellite account provide one-to-one English labels.

The downloadable `aggregation.xlsx` is already filled in for this example.
To define a different mapping, generate a template from the parsed database
**before applying aggregation**:

```python
db.get_aggregation_excel(
    path="my_aggregation.xlsx",
    levels=["Activity", "Commodity", "Consumption category",
            "Factor of production", "Satellite account"],
)
```

Fill the `Aggregation` column in each sheet with the destination labels,
then pass `io="my_aggregation.xlsx"` to `aggregate()`. Repeated destination
labels combine source items; one-to-one labels rename them. The template
does not choose the groups or translate labels automatically. A different
classification also needs compatible CVXLab set attributes. See the
[get_aggregation_excel API](https://www.marioproject.com/en/latest/api_document/mario.Database.get_aggregation_excel.html).

The template-generation step is optional. To reproduce the example, use the
completed `aggregation.xlsx` from Tutorial materials.

Apply all five sheets in one call. Imports follow the commodity aggregation;
final-demand categories, factors and satellite indicators remain separate.
The monetary unit is renamed `million EUR` without numerical rescaling.


In [ ]:
db.aggregate(
    io="aggregation.xlsx",
    levels=["Activity", "Commodity", "Consumption category",
            "Factor of production", "Satellite account"],
    calc_all=False,
    zero_output_epsilon=None,
)

for level in ["Activity", "Commodity", "Factor of production"]:
    db.units[level].loc[:, "unit"] = "million EUR"
db.meta.model_configuration["import_unit"] = "million EUR"

<a id="Explore-the-reference"></a>

## Explore the database

Use `get_index()` to inspect classification labels and `query()` to retrieve
matrices as pandas objects. Start with the activities and final-demand categories.

In [ ]:
db.get_index("Activity")

In [ ]:
db.get_index("Consumption category")

### Check matrix dimensions

`S` has activities on rows and commodities on columns. `U` reverses those axes;
`Yc` has commodities on rows and final-demand categories on columns. `IMP_a`
and `IMP_Y` keep product imports separate. The combined `Z` and `X` views retain
both productive classifications; the database remains a SUT.

In [ ]:
import pandas as pd

pd.DataFrame(
    [(name, *db.query(name).shape)
     for name in ["Z", "X", "S", "U", "Yc", "IMP_a", "IMP_Y", "Va", "Ea", "EY"]],
    columns=["Matrix", "Rows", "Columns"],
).set_index("Matrix")

### Domestic inputs, imports and value added

Compare domestic and imported intermediate purchases with the source's eight
value-added/accounting rows, in million EUR. Imports in `IMP_a` are separate
from `Va`. `IMP_Y` contains direct final-demand imports and is not included
in this activity table. Published monetary rounding differences are retained.

In [ ]:
activity_inputs = pd.DataFrame({
    "Domestic intermediate inputs": db.query("U").sum(axis=0),
    "Imported intermediate inputs": db.query("IMP_a").sum(axis=0),
    "Value added/accounting items": db.query("Va").sum(axis=0),
})
activity_inputs.index = activity_inputs.index.get_level_values("Item")
activity_inputs

### Select an activity's supply

Retrieve the products supplied by transport activities. `query()` retrieves
the complete matrix; pandas `xs()` selects the `Item` index level. To inspect
purchases instead, select the activity column of `U` with
`db.query("U").xs("Transport activities", level="Item", axis=1)`.

In [ ]:
db.query("S").xs("Transport activities", level="Item")

### Read satellite accounts with their units

The 27 rows comprise 24 AEA indicators, employed persons, hours worked and
net domestic energy use. Inspect the units before comparing their values.

In [ ]:
indicators = [
    "Greenhouse gases (CO2 equivalent) [AEA: GHG]",
    "Total employment (domestic concept)",
    "Hours worked",
    "Net domestic energy use",
]
db.units["Satellite account"].loc[indicators]

Rank activities by direct production GHG emissions in t CO2-eq. This is a
direct-flow row, not an emission intensity or footprint multiplier. `GHG` is
already characterised: do not sum it with overlapping gas components or
equivalent versions. These national accounts do not include emissions
embodied in imported goods.

In [ ]:
ghg = "Greenhouse gases (CO2 equivalent) [AEA: GHG]"
db.query("Ea").loc[ghg].sort_values(ascending=False).head()

`EY` holds direct household emissions separately from production emissions.
Inspect the same indicator across final-demand categories:

In [ ]:
db.query("EY").loc[[ghg]]

## Export to CVXLab

`cvxlab_model/model_settings.xlsx` defines the CVXLab sets, database tables,
variables and equations. `to_cvxlab()` exports the matrices of the MARIO
database's `baseline` scenario. The `mapping` argument connects their axes
to CVXLab sets and assigns each matrix to a CVXLab input table. The variables
in those tables are defined by the model settings:

| MARIO matrix | CVXLab input variable | CVXLab database table |
| --- | --- | --- |
| `S` | `S_0` | `make_0` |
| `U` | `U_0` | `use_0` |
| `Yc` | `Y_0` | `use_0` |
| `IMP_a` | `IMP_a_0` | `imports_0` |
| `IMP_Y` | `IMP_y_0` | `imports_0` |
| `Va` | `V_a_0` | `extensions_0` |
| `Ea` | `E_a_0` | `extensions_0` |
| `EY` | `E_y_0` | `extensions_0` |
| `Xa` | `X_a_0` | `total_activity_0` |
| `Xc` | `X_c_0` | `total_demand_0` |

The `_0` suffix identifies input variables in this model configuration.
These six input tables contain the baseline values exported from MARIO.
Several variables share a table: `U_0` and `Y_0` select intermediate and
final-demand columns from `use_0`; the import variables use the same split
in `imports_0`. Within `extensions_0`, `V_a_0` selects value-added rows,
while `E_a_0` and `E_y_0` select satellite rows for production and final
demand. These selections use the category attributes in `sets.xlsx`.

Activities and final demand share the `accounts` set; factors and satellites
share `extensions`. This configuration has no exogenous input for `VY`;
that final-demand accounting matrix remains in the MARIO database.

### CVXLab scenario inputs

`model_settings.xlsx` also declares `delta_make`, `delta_use` and
`delta_imports` as scenario-dependent input tables. `to_cvxlab()` creates
their templates, but leaves their values blank because they have no MARIO
matrix mapping. They therefore appear in `report["unmapped_tables"]`.
When CVXLab loads them, the configuration's `blank_fill=0` supplies zeros.
MARIO does not infer that the changes are zero.

In `sets.xlsx`, `Reference` is the name of the CVXLab scenario for the
zero-change case. `baseline` is the MARIO scenario being exported. The
six `_0` input tables do not have a CVXLab scenario dimension; the
`delta_*` tables do.

### Write CVXLab inputs

`to_cvxlab()` writes the classified sets to `sets.xlsx` and the input tables
to `input_data/input_data.xlsx` inside `cvxlab_model/`. Load these files
into CVXLab using the accompanying model configuration.



In [ ]:
report = db.to_cvxlab(
    model_settings="cvxlab_model/model_settings.xlsx",
    path="cvxlab_model",
    mapping={
        "sets": {
            "Commodity_from": "products", "Commodity_to": "products",
            "Activity_from": "accounts", "Activity_to": "accounts",
            "Consumption category_to": "accounts",
            "Factor of production_from": "extensions",
            "Satellite account_from": "extensions",
        },
        "matrices": {
            "S": "make_0", "U": "use_0", "Yc": "use_0",
            "IMP_a": "imports_0", "IMP_Y": "imports_0",
            "Va": "extensions_0", "Ea": "extensions_0", "EY": "extensions_0",
            "Xa": "total_activity_0", "Xc": "total_demand_0",
        },
    },
)
report["written_cells"]